# ADPKD Outcome Model (ADPKD_OM)

Ce notebook reproduit l'approche de l'**ADPKD Outcome Model** sur les données de la cohorte **Genkyst** afin d'étudier la progression de la polykystose rénale autosomique dominante (PKRAD).

Il comprend la préparation des données longitudinales, l'analyse des trajectoires du TKV et du DFG, ainsi que l'ajustement de modèles linéaires mixtes pour évaluer l'association entre le volume rénal initial, les caractéristiques des patients et l'évolution de la maladie.


In [ ]:
# ==========================================================
# LIBRAIRIES
# ==========================================================

# Gestion des fichiers
import os
from pathlib import Path

# Calcul et manipulation des données
import numpy as np
import pandas as pd
import polars as pl

# Visualisation
import plotly.express as px

# Analyses statistiques
from scipy.stats import kruskal, spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Gestion des avertissements
import warnings

warnings.filterwarnings("ignore")

## CHARGEMENT DES DONNEES 

#### Inclusion

In [ ]:
# ==========================================================
# CHARGEMENT DES DONNÉES
# ==========================================================

inclusion = pl.read_parquet("Preprocessed_data/inclusion.parquet")
image = pl.read_parquet("Preprocessed_data/image.parquet")
creat = pl.read_parquet("Preprocessed_data/creat.parquet")

# ==========================================================
# APERÇU DU JEU DE DONNÉES D'INCLUSION
# ==========================================================

print(f"Taille du jeu de données : {inclusion.shape}")
print(
    f"Nombre de patients uniques : "
    f"{inclusion.select(pl.col('patient_id').n_unique()).item()}"
)

# Pourcentage de valeurs manquantes par variable
inclusion.null_count() / len(inclusion) * 100

In [ ]:
# ==========================================================
# CALCUL DU VOLUME RÉNAL TOTAL (TKV)
# ==========================================================

image = image.with_columns(
    pl.when(
        pl.col("left_kidney_volume").is_not_null()
        & pl.col("right_kidney_volume").is_not_null()
    )
    .then(
        pl.col("left_kidney_volume")
        + pl.col("right_kidney_volume")
    )
    .otherwise(None)
    .alias("TKV")
)

# ==========================================================
# APERÇU DU JEU DE DONNÉES D'IMAGERIE
# ==========================================================

print(f"Taille du jeu de données : {image.shape}")
print(
    f"Nombre de patients uniques : "
    f"{image.select(pl.col("patient_id").n_unique()).item()}"
)

# Pourcentage de valeurs manquantes par variable
image.null_count() / len(image) * 100

In [ ]:
# ==========================================================
# PRÉPARATION DES DONNÉES D'IMAGERIE
# ==========================================================

# Ajout des informations d'inclusion
image_merged = image.join(
    inclusion.select(["patient_id", "birth_date", "sex"]),
    on="patient_id",
    how="left"
)

# Calcul de l'âge au moment de l'examen d'imagerie
image_with_age = (
    image_merged
    .filter(pl.col("birth_date") <= pl.col("image_date"))
    .with_columns(
        (
            (pl.col("image_date") - pl.col("birth_date"))
            .dt.total_days() / 365.25
        ).alias("age")
    )
)

# Conservation des observations exploitables
image_clean = image_with_age.filter(
    pl.col("TKV").is_not_null()
    & pl.col("age").is_not_null()
)

# ==========================================================
# APERÇU DU JEU DE DONNÉES FINAL
# ==========================================================

print(f"Taille du jeu de données : {image_clean.shape}")
print(
    f"Nombre de patients uniques : "
    f"{image_clean.select(pl.col('patient_id').n_unique()).item()}"
)

# Pourcentage de valeurs manquantes par variable
image_clean.null_count() / len(image_clean) * 100

In [ ]:
# ==========================================================
# SÉLECTION DES VARIABLES D'INTÉRÊT
# ==========================================================

cols_to_keep = [
    "patient_id",
    "image_date",
    "birth_date",
    "age",
    "TKV",
    "class_mayo",
    "sex",
]

image_model = image_clean.select(
    [col for col in cols_to_keep if col in image_clean.columns]
)

# ==========================================================
# APERÇU DU JEU DE DONNÉES FINAL
# ==========================================================

print(f"Taille du jeu de données : {image_model.shape}")
print(
    f"Nombre de patients uniques : "
    f"{image_model.select(pl.col('patient_id').n_unique()).item()}"
)

display(image_model)

# Pourcentage de valeurs manquantes par variable
image_model.null_count() / len(image_model) * 100

La variable class_mayo comportait 1.44 % de valeurs manquantes, considérées comme négligeables et exclues des analyses nécessitant cette variable.

In [ ]:
# ==========================================================
# EXAMEN DES PATIENTS AYANT UN TKV < 200 mL
# ==========================================================

image_model.filter(
    pl.col("TKV") < 200
)

Quelques valeurs de TKV inférieures à 200 mL ont été observées (moins de 1 % des observations). Après vérification, elles ont été jugées plausibles et ont donc été conservées pour les analyses.


In [ ]:
# ==========================================================
# TRANSFORMATION LOGARITHMIQUE DU TKV
# ==========================================================

image_model = image_model.with_columns(
    pl.col("TKV").log().alias("log_TKV")
)

# Aperçu des premières observations
print(image_model.head())

# Statistiques descriptives de la variable transformée
image_model.select("log_TKV").describe()

In [ ]:
# ==========================================================
# SÉLECTION DE L'EXAMEN D'IMAGERIE INITIAL
# ==========================================================

# Conservation de la première imagerie disponible pour chaque patient
image_baseline = (
    image_model
    .sort(["patient_id", "image_date"])
    .group_by("patient_id")
    .first()
)

# Conservation des patients ayant une classification de Mayo
image_mayo = image_baseline.filter(
    pl.col("class_mayo").is_not_null()
)

# ==========================================================
# APERÇU DU JEU DE DONNÉES FINAL
# ==========================================================

print(f"Taille du jeu de données : {image_mayo.shape}")
print(
    f"Nombre de patients uniques : "
    f"{image_mayo.select(pl.col('patient_id').n_unique()).item()}"
)

# Pourcentage de valeurs manquantes par variable
image_mayo.null_count() / len(image_mayo) * 100

In [ ]:
# ==========================================================
# IDENTIFICATION DES PATIENTS TYPIQUES (MAYO 1A–1E)
# ==========================================================

# Patients typiques à l'examen d'imagerie initial
typical_baseline = image_baseline.filter(
    pl.col("class_mayo").is_in(["1A", "1B", "1C", "1D", "1E"])
)

# Liste des patients typiques
typical_patients = typical_baseline.select("patient_id").unique()

# Toutes les observations longitudinales des patients typiques
image_long_typical = image_model.join(
    typical_patients,
    on="patient_id",
    how="inner"
)

# ==========================================================
# RÉSUMÉ DES DONNÉES
# ==========================================================

print("=== Cohorte baseline ===")
print(
    f"Nombre de patients : "
    f"{typical_baseline.select(pl.col('patient_id').n_unique()).item()}"
)
print(f"Nombre d'observations : {typical_baseline.height}")

print("\n=== Cohorte longitudinale ===")
print(
    f"Nombre de patients : "
    f"{image_long_typical.select(pl.col('patient_id').n_unique()).item()}"
)
print(f"Nombre d'observations : {image_long_typical.height}")

In [ ]:
# ==========================================================
# STATISTIQUES DESCRIPTIVES DU log(TKV) PAR CLASSE DE MAYO
# ==========================================================

summary_stats = (
    typical_baseline
    .group_by("class_mayo")
    .agg([
        pl.len().alias("n"),
        pl.col("log_TKV").median().alias("median"),
        pl.col("log_TKV").quantile(0.25).alias("q25"),
        pl.col("log_TKV").quantile(0.75).alias("q75"),
    ])
    .with_columns(
        (pl.col("q75") - pl.col("q25")).alias("IQR")
    )
    .sort("class_mayo")
)

summary_stats

**Interprétation :**

Une augmentation progressive du log(TKV) est observée de la classe Mayo 1A à la classe 1E. La médiane passe de **5,98** pour la classe 1A à **8,07** pour la classe 1E, ce qui indique que les patients classés dans les catégories les plus sévères présentent, en moyenne, un volume rénal plus élevé. La dispersion des valeurs (IQR) est également plus importante dans les classes avancées, traduisant une plus grande variabilité du volume rénal chez ces patients. Ces résultats sont cohérents avec la classification de Mayo, qui repose notamment sur le volume rénal total ajusté à l'âge.


#### Caractéristiques basales 

In [ ]:
# ==========================================================
# ASSOCIATION DES MESURES DE DFG AUX IMAGERIES BASELINE
# ==========================================================

# Ajout de toutes les mesures de DFG disponibles pour chaque patient
baseline_with_egfr = (
    typical_baseline
    .join(
        creat.select(["patient_id", "ckd_epi", "creat_date"]),
        on="patient_id",
        how="left"
    )
    .filter(pl.col("ckd_epi").is_not_null())
)

# ==========================================================
# APERÇU DU JEU DE DONNÉES
# ==========================================================

print(f"Taille du jeu de données : {baseline_with_egfr.shape}")
print(
    f"Nombre de patients uniques : "
    f"{baseline_with_egfr.select(pl.col('patient_id').n_unique()).item()}"
)

baseline_with_egfr.head()

# Pourcentage de valeurs manquantes par variable
baseline_with_egfr.null_count() / len(baseline_with_egfr) * 100

In [ ]:
# ==========================================================
# SÉLECTION DE LA MESURE DE DFG LA PLUS RÉCENTE AVANT L'IMAGERIE
# ==========================================================

baseline_final = (
    baseline_with_egfr
    # Conserver uniquement les mesures réalisées avant ou le jour de l'imagerie
    .filter(pl.col("creat_date") <= pl.col("image_date"))
    
    # Trier les mesures par patient, de la plus récente à la plus ancienne
    .sort(["patient_id", "creat_date"], descending=[False, True])
    
    # Conserver la mesure de DFG la plus récente avant l'imagerie
    .group_by("patient_id")
    .first()
)

In [ ]:
# ==========================================================
# CARACTÉRISTIQUES DESCRIPTIVES DE LA COHORTE BASELINE
# ==========================================================

summary_table = pl.DataFrame({
    "Characteristic": [
        "Male, no. (%)",
        "Age (years)",
        "TKV (mL)",
        "eGFR (mL/min/1.73m²)"
    ],
    "Overall": [
        f"{(baseline_final['sex'] == 'Male').sum()} "
        f"({((baseline_final['sex'] == 'Male').sum() / len(baseline_final) * 100):.1f})",

        f"{baseline_final['age'].mean():.1f} ± {baseline_final['age'].std():.1f}",

        f"{baseline_final['TKV'].mean():.0f} ± {baseline_final['TKV'].std():.0f}",

        f"{baseline_final['ckd_epi'].mean():.1f} ± {baseline_final['ckd_epi'].std():.1f}"
    ]
})

summary_table

In [ ]:
# ==========================================================
# EXPORT DU TABLEAU DES CARACTÉRISTIQUES AU FORMAT HTML
# ==========================================================

df_html = summary_table.to_pandas()

styled_table = (
    df_html.style
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("font-weight", "bold"),
                ("text-align", "left"),
                ("border-bottom", "2px solid black"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "left"),
                ("padding", "6px"),
            ],
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("font-family", "Arial"),
                ("width", "50%"),
            ],
        },
    ])
    .hide(axis="index")
)

html_content = f"""
<html>
<body>

<h3>Baseline characteristics of the study cohort</h3>

{styled_table.to_html()}

</body>
</html>
"""

output_path = (
    r"."
    r"\tables\adpkd_om\baseline_characteristics.html"
)

with open(output_path, "w") as f:
    f.write(html_content)

#### Cohérence de la classification Mayo

Une vérification de la cohérence de la classification Mayo au cours du suivi a été réalisée, afin d’identifier les patients présentant des variations de classe.

In [ ]:
# ==========================================================
# IDENTIFICATION DES CHANGEMENTS DE CLASSE DE MAYO
# ==========================================================

# Nombre de classes Mayo différentes observées pour chaque patient
mayo_variation = (
    image_long_typical
    .group_by("patient_id")
    .agg(
        pl.col("class_mayo").n_unique().alias("n_class")
    )
    .filter(pl.col("n_class") > 1)
)

# Nombre de patients ayant changé de classe Mayo
n_variants = mayo_variation.height

print(f"Nombre de patients ayant changé de classe Mayo : {n_variants}")

# Observations longitudinales des patients concernés
image_long_typical.filter(
    pl.col("patient_id").is_in(mayo_variation["patient_id"])
)

**Interprétation :**

Parmi les 535 patients typiques, 50 (9,3 %) ont présenté un changement de classe Mayo au cours du suivi. Cette proportion étant faible, la classification de Mayo retenue pour les analyses correspond à celle observée lors de l'examen d'imagerie initial (baseline), afin de garantir une définition cohérente de la cohorte tout au long de l'étude.


#### Analyse de log(TKV) vs eGFR

In [ ]:
# ==========================================================
# RELATION ENTRE LE log(TKV) ET LE DFG AU BASELINE
# ==========================================================

df_plot = baseline_final.to_pandas()

fig = px.scatter(
    df_plot,
    x="log_TKV",
    y="ckd_epi",
    opacity=0.6
)

fig.update_layout(
    title="Relation entre le log(TKV) et le DFG au baseline",
    xaxis_title="log(TKV)",
    yaxis_title="DFG (mL/min/1.73 m²)"
)

fig.show()

Une relation faible et dispersée est observée entre le log(TKV) et l’eGFR à l’inclusion.

#### Calcul des pentes

On estime les pentes de progression séparément pour le TKV et l’eGFR à partir de leurs bases longitudinales respectives, afin de préserver la dynamique temporelle propre à chaque variable.

##### Pentes log_TKV

In [ ]:
# ==========================================================
# CALCUL DE LA PENTE D'ÉVOLUTION DU log(TKV)
# ==========================================================

# Temps écoulé (en années) depuis la première imagerie de chaque patient
image_long_typical = image_long_typical.with_columns(
    (
        (pl.col("image_date") - pl.col("image_date").min().over("patient_id"))
        .dt.total_days() / 365.25
    ).alias("time_years")
)

# Estimation de la pente individuelle du log(TKV)
slope_tkv = (
    image_long_typical
    .group_by("patient_id")
    .agg([
        # Nombre de mesures de TKV
        pl.len().alias("n_tkv"),

        # Durée de suivi (années)
        (
            pl.col("time_years").max()
            - pl.col("time_years").min()
        ).alias("time_span"),

        # Pente estimée par régression linéaire simple
        (
            pl.cov("time_years", "log_TKV")
            / pl.var("time_years")
        ).alias("slope_log_TKV")
    ])
    .filter(
        (pl.col("n_tkv") >= 2) &
        (pl.col("time_span") > 0.5)
    )
)

# ==========================================================
# EXCLUSION DES VALEURS EXTRÊMES (MÉTHODE DE L'IQR)
# ==========================================================

q1 = slope_tkv["slope_log_TKV"].quantile(0.25)
q3 = slope_tkv["slope_log_TKV"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

slope_tkv = slope_tkv.filter(
    (pl.col("slope_log_TKV") >= lower) &
    (pl.col("slope_log_TKV") <= upper)
)

# ==========================================================
# STATISTIQUES DESCRIPTIVES DES PENTES
# ==========================================================

slope_tkv.describe()

Les valeurs extrêmes de pente ont été exclues selon la méthode de l’intervalle interquartile (IQR), permettant de limiter l’impact des valeurs aberrantes tout en conservant la majorité de la cohorte.

##### Pentes eGFR

Les pentes d’eGFR ont été calculées à partir des mesures réalisées sur la même période que les examens d’imagerie, afin d’assurer une comparaison cohérente avec le TKV.

In [ ]:
# ==========================================================
# SÉLECTION DES MESURES DE DFG PENDANT LE SUIVI D'IMAGERIE
# ==========================================================

# Définition de la période de suivi en imagerie pour chaque patient
image_window = (
    image_long_typical
    .group_by("patient_id")
    .agg([
        pl.col("image_date").min().alias("start_date"),
        pl.col("image_date").max().alias("end_date")
    ])
)

# Conservation des mesures de DFG réalisées pendant cette période
creat_window = (
    creat
    .join(image_window, on="patient_id", how="inner")
    .filter(
        (pl.col("creat_date") >= pl.col("start_date")) &
        (pl.col("creat_date") <= pl.col("end_date"))
    )
)

# Préparation du jeu de données longitudinal
creat_long = (
    creat_window
    .select(["patient_id", "creat_date", "ckd_epi"])
    .filter(
        pl.col("ckd_epi").is_not_null() &
        pl.col("creat_date").is_not_null()
    )
    .sort(["patient_id", "creat_date"])
)

In [ ]:
# ==========================================================
# CALCUL DE LA PENTE D'ÉVOLUTION DU DFG
# ==========================================================

# Temps écoulé (années) depuis la première mesure de DFG
creat_long = creat_long.with_columns(
    (
        (pl.col("creat_date") - pl.col("creat_date").min().over("patient_id"))
        .dt.total_days() / 365.25
    ).alias("time_years")
)

# Estimation de la pente individuelle du DFG
slope_egfr = (
    creat_long
    .group_by("patient_id")
    .agg([
        # Nombre de mesures de DFG
        pl.len().alias("n_egfr"),

        # Durée de suivi (années)
        (
            pl.col("time_years").max()
            - pl.col("time_years").min()
        ).alias("time_span"),

        # Pente estimée par régression linéaire simple
        (
            pl.cov("time_years", "ckd_epi")
            / pl.var("time_years")
        ).alias("slope_eGFR")
    ])
    .filter(
        (pl.col("n_egfr") >= 2) &
        (pl.col("time_span") > 0.5)
    )
)

In [ ]:
# ==========================================================
# EXCLUSION DES VALEURS EXTRÊMES ET RÉSUMÉ DES PENTES
# ==========================================================

# Détection des valeurs extrêmes par la méthode de l'IQR
q1 = slope_egfr["slope_eGFR"].quantile(0.25)
q3 = slope_egfr["slope_eGFR"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# Conservation des pentes non atypiques
slope_egfr = slope_egfr.filter(
    (pl.col("slope_eGFR") >= lower) &
    (pl.col("slope_eGFR") <= upper)
)

# Statistiques descriptives des pentes
slope_egfr.describe()

**Interprétation :**

Les valeurs extrêmes des pentes de DFG ont été identifiées à l'aide de la méthode de l'intervalle interquartile (IQR) puis exclues, afin de limiter l'influence de valeurs atypiques sur les analyses tout en conservant la majorité de la cohorte.

In [ ]:
# ==========================================================
# CONSTITUTION DE LA BASE FINALE DES PENTES
# ==========================================================

# Conservation des patients disposant d'une estimation de pente
# pour le log(TKV) et le DFG
slopes_final = slope_tkv.join(
    slope_egfr,
    on="patient_id",
    how="inner"
)

# ==========================================================
# STATISTIQUES DESCRIPTIVES DE LA BASE FINALE
# ==========================================================

slopes_final.describe()

**Interprétation :**

Après préparation des données longitudinales, les pentes de progression ont été estimées séparément pour le volume rénal total (TKV) et le DFG, afin de préserver la dynamique temporelle propre à chaque variable. Les pentes du log(TKV) ont été calculées à partir des données d'imagerie répétées, tandis que les pentes du DFG ont été estimées à partir des mesures longitudinales de créatinine, en ajustant pour chaque patient une régression linéaire en fonction du temps écoulé depuis la première mesure. Les mesures de DFG ont été restreintes à la période couverte par les examens d'imagerie afin d'assurer une cohérence temporelle avec les données de TKV.

Afin de garantir la robustesse des estimations, seules les trajectoires comportant au moins deux mesures et une durée de suivi supérieure à six mois ont été conservées. Les valeurs extrêmes des pentes ont ensuite été identifiées à l'aide de la méthode de l'intervalle interquartile (IQR) et exclues afin de limiter l'influence de valeurs atypiques.

Au terme de ces étapes de sélection, **105 patients** disposaient d'une estimation fiable des pentes du log(TKV) et du DFG. La croissance médiane du log(TKV) était de **0,035/an** (IQR : **0,010–0,056**), tandis que le déclin médian du DFG était de **−3,31 mL/min/1,73 m²/an** (IQR : **−5,32 à −1,61**). Ces résultats mettent en évidence une progression globalement défavorable de la maladie, avec une augmentation du volume rénal associée à une diminution de la fonction rénale, tout en soulignant l'importante variabilité interindividuelle des vitesses de progression.


### Relation entre le volume rénal et le déclin de la fonction rénale

Après avoir estimé les pentes individuelles d'évolution du log(TKV) et du DFG, l'objectif est d'examiner l'association entre la progression du volume rénal et celle de la fonction rénale. Cette analyse vise à déterminer si une croissance plus rapide du volume rénal est associée à un déclin plus marqué du DFG, conformément à l'histoire naturelle de la polykystose rénale autosomique dominante (PKRAD).


In [ ]:
# ==========================================================
# RELATION ENTRE LE TKV AU BASELINE ET LA PENTE DE DÉCLIN DU DFG
# ==========================================================

# Ajout du log(TKV) et de la classe de Mayo au jeu de données
analysis_df = slopes_final.join(
    typical_baseline.select(["patient_id", "log_TKV", "class_mayo"]),
    on="patient_id",
    how="inner"
)

# Nuage de points avec droite de régression
fig = px.scatter(
    analysis_df.to_pandas(),
    x="log_TKV",
    y="slope_eGFR",
    trendline="ols"
)

fig.update_layout(
    title="Relation entre le log(TKV) au baseline et la pente de déclin du DFG",
    xaxis_title="log(TKV) au baseline",
    yaxis_title="Pente du DFG (mL/min/1,73 m²/an)"
)

fig.show()

In [ ]:
# ==========================================================
# ASSOCIATION ENTRE LE log(TKV) AU BASELINE ET LA PENTE DU DFG
# ==========================================================

# Corrélation de Spearman
spearmanr(
    analysis_df["log_TKV"],
    analysis_df["slope_eGFR"]
)

**Interprétation :**

Une corrélation de Spearman a mis en évidence une association négative modérée et statistiquement significative entre le log(TKV) au baseline et la pente du DFG (ρ = −0,337 ; *p* < 0,001). Les patients présentant un volume rénal initial plus élevé avaient tendance à présenter un déclin plus rapide de la fonction rénale au cours du suivi. Bien que cette corrélation soit d'intensité modérée, elle est cohérente avec l'histoire naturelle de la polykystose rénale autosomique dominante (PKRAD), dans laquelle l'augmentation du volume rénal constitue un marqueur précoce de la progression de la maladie et précède généralement la détérioration de la fonction rénale.


### Déclin de la fonction rénale selon la classification Mayo

Afin d'évaluer l'association entre la classification de Mayo et la progression fonctionnelle de la maladie, les pentes individuelles de déclin du DFG ont été comparées entre les différentes classes de Mayo. Cette analyse vise à déterminer si les patients appartenant aux classes de Mayo les plus avancées présentent un déclin plus rapide de la fonction rénale, conformément à la stratification pronostique proposée par cette classification.


In [ ]:
# ==========================================================
# PENTE DE DÉCLIN DU DFG SELON LA CLASSIFICATION DE MAYO
# ==========================================================

# Distribution des pentes de déclin du DFG selon la classe de Mayo
fig = px.box(
    analysis_df.to_pandas(),
    x="class_mayo",
    y="slope_eGFR",
    color="class_mayo",
    category_orders={"class_mayo": ["1A", "1B", "1C", "1D", "1E"]},
    points="all"
)

fig.update_layout(
    title="Pente de déclin du DFG selon la classification de Mayo",
    xaxis_title="Classe de Mayo",
    yaxis_title="Pente du DFG (mL/min/1,73 m²/an)",
    showlegend=False
)

fig.show()

In [ ]:
# ==========================================================
# STATISTIQUES DESCRIPTIVES PAR CLASSE DE MAYO
# ==========================================================

# Effectif, médiane et intervalle interquartile (IQR)
# des pentes de déclin du DFG selon la classification de Mayo
summary_mayo = (
    analysis_df
    .group_by("class_mayo")
    .agg([
        pl.len().alias("n"),

        pl.col("slope_eGFR").median().alias("median"),
        pl.col("slope_eGFR").quantile(0.25).alias("q25"),
        pl.col("slope_eGFR").quantile(0.75).alias("q75"),
    ])
    .with_columns(
        (pl.col("q75") - pl.col("q25")).alias("IQR")
    )
    .sort("class_mayo")
)

summary_mayo

In [ ]:
# ==========================================================
# COMPARAISON DES PENTES DE DÉCLIN DU DFG ENTRE LES CLASSES DE MAYO
# ==========================================================

# Test non paramétrique de Kruskal-Wallis
kruskal(
    *[
        analysis_df.filter(pl.col("class_mayo") == c)["slope_eGFR"]
        for c in ["1A", "1B", "1C", "1D", "1E"]
    ]
)

**Interprétation :**

Le test de Kruskal-Wallis met en évidence une différence statistiquement significative des pentes de déclin du DFG entre les classes de Mayo (H = 20,52 ; *p* < 0,001). Les médianes des pentes montrent une aggravation progressive du déclin de la fonction rénale entre les classes 1A et 1D, passant de −1,09 à −4,86 mL/min/1,73 m²/an. Cette tendance est cohérente avec la classification de Mayo, selon laquelle les classes les plus élevées correspondent à une progression plus rapide de la maladie.

La classe 1E présente une médiane de déclin moins marquée que la classe 1D. Toutefois, cette estimation doit être interprétée avec prudence en raison du faible effectif de cette classe (n = 9), susceptible d'entraîner une plus grande variabilité des estimations.

Dans l'ensemble, ces résultats confirment que la classification de Mayo est associée à des différences significatives de vitesse de déclin du DFG chez les patients atteints de polykystose rénale autosomique dominante. Des comparaisons post hoc seraient néanmoins nécessaires pour identifier précisément quelles classes diffèrent significativement les unes des autres.


#### Modélisation de la progression du TKV et du déclin du DFG selon les caractéristiques de base

Afin d'étudier l'influence des caractéristiques initiales sur la progression de la maladie, nous avons adopté une approche inspirée de l'**ADPKD Outcome Model** en estimant séparément deux modèles de régression linéaire multivariée : l'un pour la croissance du volume rénal total (TKV) et l'autre pour le déclin de la fonction rénale (DFG).

Le premier modèle a pour variable dépendante la pente annuelle de croissance du log(TKV), tandis que le second modélise la pente annuelle de déclin du DFG. Dans les deux modèles, les variables explicatives sont l'âge au baseline, le log(TKV) au baseline et le sexe. Conformément à l'approche retenue dans l'ADPKD Outcome Model, un terme d'interaction entre l'âge et le log(TKV) a été ajouté afin de tenir compte de leur effet conjoint sur la progression de la maladie.


#### **Modèle 1 (TKV)**

In [ ]:
# ==========================================================
# PRÉPARATION DES DONNÉES POUR LE MODÈLE MIXTE DU log(TKV)
# ==========================================================

# Variables nécessaires à la modélisation
df_long = image_long_typical.select([
    "patient_id",
    "image_date",
    "log_TKV",
    "age",
    "sex"
])

# ==========================================================
# ENCODAGE DES VARIABLES
# ==========================================================

# Encodage binaire du sexe (1 = homme, 0 = femme)
df_long = df_long.with_columns(
    (pl.col("sex") == "Male").cast(pl.Int8).alias("male")
)

# ==========================================================
# CALCUL DU log(TKV) AU BASELINE
# ==========================================================

baseline_tkv = (
    df_long
    .sort(["patient_id", "image_date"])
    .group_by("patient_id")
    .agg(
        pl.col("log_TKV").first().alias("log_TKV_baseline")
    )
)

# Ajout du log(TKV) au baseline à chaque visite
df_long = df_long.join(
    baseline_tkv,
    on="patient_id",
    how="left"
)

# ==========================================================
# CRÉATION DU TERME D'INTERACTION
# ==========================================================

df_long = df_long.with_columns(
    (pl.col("age") * pl.col("log_TKV_baseline")).alias("age_log_TKV")
)

# ==========================================================
# SÉLECTION DES PATIENTS AYANT AU MOINS TROIS MESURES
# ==========================================================

df_long = df_long.with_columns(
    pl.len().over("patient_id").alias("n_measures")
)

df_long_filtered = df_long.filter(
    pl.col("n_measures") >= 3
)

# ==========================================================
# AJUSTEMENT DU MODÈLE MIXTE LINÉAIRE
# ==========================================================

# Conversion en pandas (requise par statsmodels)
df = df_long_filtered.to_pandas()

model_tkv = smf.mixedlm(
    "log_TKV ~ age + log_TKV_baseline + male + age_log_TKV",
    df,
    groups=df["patient_id"]
)

result_tkv = model_tkv.fit()

# Résumé des coefficients estimés
print(result_tkv.summary())

In [ ]:
# ==========================================================
# NOMBRE DE MESURES PAR PATIENT
# ==========================================================

image_long_typical.group_by("patient_id").len().describe()

In [ ]:
# ==========================================================
# DURÉE DE SUIVI PAR PATIENT
# ==========================================================

image_long_typical.group_by("patient_id").agg(
    (pl.col("time_years").max() - pl.col("time_years").min()).alias("time_span")
).describe()

**Interprétation :**

Le modèle linéaire mixte met en évidence une association positive et hautement significative entre le log(TKV) au baseline et la croissance ultérieure du TKV (β = 0,72 ; *p* < 0,001). Les patients présentant un volume rénal initial plus élevé avaient ainsi tendance à présenter une progression plus rapide du volume rénal au cours du suivi, ce qui est cohérent avec les connaissances actuelles sur l'histoire naturelle de la polykystose rénale autosomique dominante.

En revanche, ni l'âge ni le terme d'interaction entre l'âge et le log(TKV) au baseline n'atteignent le seuil de significativité statistique, bien qu'une tendance soit observée (*p* = 0,10 et *p* = 0,08, respectivement).

Ces résultats doivent toutefois être interprétés avec prudence. La cohorte d'imagerie disponible comporte relativement peu de mesures longitudinales par patient (médiane : 1 examen ; intervalle interquartile : 1–2) et une durée de suivi souvent limitée (médiane : 0 an ; intervalle interquartile : 0–1,49 an). Cette faible densité des données longitudinales peut limiter la précision des estimations et la capacité du modèle à caractériser pleinement la dynamique de croissance du TKV.


#### **Modèle 2 (eGFR)**

In [ ]:
# ==========================================================
# PRÉPARATION DES DONNÉES POUR LE MODÈLE MIXTE DU DFG
# ==========================================================

# Variables longitudinales du DFG
df_egfr = creat_long.select([
    "patient_id",
    "creat_date",
    "ckd_epi",
    "time_years"
])

# ==========================================================
# AJOUT DES VARIABLES DE BASELINE
# ==========================================================

# Ajout du log(TKV) au baseline
df_egfr = df_egfr.join(
    baseline_tkv,
    on="patient_id",
    how="inner"
)

# Ajout de l'âge et du sexe au baseline
df_egfr = df_egfr.join(
    typical_baseline.select(["patient_id", "age", "sex"]),
    on="patient_id",
    how="left"
)

# ==========================================================
# ENCODAGE DES VARIABLES
# ==========================================================

# Encodage binaire du sexe (1 = homme, 0 = femme)
df_egfr = df_egfr.with_columns(
    (pl.col("sex") == "Male").cast(pl.Int8).alias("male")
)

# ==========================================================
# CRÉATION DU TERME D'INTERACTION
# ==========================================================

df_egfr = df_egfr.with_columns(
    (pl.col("age") * pl.col("log_TKV_baseline")).alias("age_log_TKV")
)

# ==========================================================
# SÉLECTION DES PATIENTS AYANT AU MOINS TROIS MESURES
# ==========================================================

df_egfr = df_egfr.filter(
    pl.len().over("patient_id") >= 3
)

# ==========================================================
# AJUSTEMENT DU MODÈLE MIXTE LINÉAIRE
# ==========================================================

# Conversion en pandas (requise par statsmodels)
df = df_egfr.to_pandas()

model_egfr = smf.mixedlm(
    "ckd_epi ~ time_years + log_TKV_baseline + age + male + age_log_TKV",
    df,
    groups=df["patient_id"]
)

result_egfr = model_egfr.fit()

# Résumé des coefficients estimés
print(result_egfr.summary())

**Interprétation :**

Le modèle linéaire mixte met en évidence un déclin significatif du DFG au cours du suivi, avec une diminution moyenne estimée à **3,47 mL/min/1,73 m² par an** (*β* = −3,47 ; *p* < 0,001).

Le log(TKV) au baseline est également significativement associé au DFG au cours du suivi (*β* = −24,66 ; *p* = 0,022). Ainsi, à caractéristiques égales, les patients présentant un volume rénal initial plus élevé avaient tendance à présenter des valeurs de DFG plus faibles au cours de leur suivi.

En revanche, l'âge, le sexe et le terme d'interaction entre l'âge et le log(TKV) au baseline n'étaient pas significativement associés au DFG dans ce modèle (*p* > 0,05).

Dans l'ensemble, ces résultats mettent en évidence une association entre un volume rénal initial élevé et une fonction rénale plus altérée au cours du suivi. Ils confirment également le déclin progressif du DFG chez les patients atteints de polykystose rénale autosomique dominante, en accord avec les connaissances actuelles sur l'évolution naturelle de la maladie.
